###  Metadata builder to retrieve filter bottoms from filename

In [1]:
# Setup: Import modules and define paths
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Define folders
wiertsema_input_dir = repo_root / 'output_data' / 'only_csv_wiertsema'
fugro_input_dir = repo_root / 'output_data' / 'only_csv_fugro'

print('Setup complete!')
print(f'Repo root: {repo_root}')
print(f'Wiertsema input: {wiertsema_input_dir}')
print(f'Fugro input: {fugro_input_dir}')

Setup complete!
Repo root: d:\Users\jvanruitenbeek\data_validation
Wiertsema input: d:\Users\jvanruitenbeek\data_validation\output_data\only_csv_wiertsema
Fugro input: d:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro


In [2]:
import re
import pandas as pd
from pathlib import Path

# Folders (as you already have)
wiertsema_input_dir = repo_root / 'output_data' / 'only_csv_wiertsema'
fugro_input_dir      = repo_root / 'output_data' / 'only_csv_fugro'

# --- Extraction helpers ---
fugro_pattern = re.compile(
    r"_([+-]?\d+(?:\.\d+)?)_m_NAP_avg\.csv$", re.IGNORECASE
)

def extract_fugro_depth(name: str):
    """
    Extract Fugro bottom filter as the float directly before '_m_NAP_avg'.
    Returns float or None.
    """
    m = fugro_pattern.search(name)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return None


# Wiertsema examples:

#FIXME 
#FIXME 
#FIXME# SOMMIGE WIERTSEMA BESTANDEN HEBBEN EEN PUNT IN PLAATS VAN EEN KOMMA OF GEEN PUNT CODE AANPASSEN


import re

wiertsema_pattern = re.compile(r"_F(-[0-9]+(?:\.[0-9]+)?)")

def extract_wiertsema_depth(name: str):
    """
    Extract Wiertsema filter bottom depth (in meters).
    """

    m = wiertsema_pattern.search(name)
    if not m:
        return None

    raw_val = m.group(1)   # Now includes the minus sign!

    # Clean decimal comma if present
    raw_val_clean = raw_val.replace(",", ".")

    try:
        if "." in raw_val_clean:
            # Already in meters (negative included)
            mid = float(raw_val_clean)
        else:
            # Centimeters to meters (preserve sign)
            mid = float(raw_val_clean) / 100.0

        bottom = mid - 0.5
        return bottom

    except ValueError:
        return None


# --- Folder scanner ---

def scan_folder(folder: Path, kind: str):
    """
    Scan a folder and return rows with filename + bk_fil_ok_fil.
    kind: 'fugro' or 'wiertsema'
    """
    rows = []
    for f in sorted(folder.glob("*.csv")):
        if kind == "fugro":
            depth = extract_fugro_depth(f.name)
        elif kind == "wiertsema":
            depth = extract_wiertsema_depth(f.name)
        else:
            depth = None

        # If we can't find a valid bottom filter (bot_fil), keep it empty.
        rows.append({
            "filename": f.name,
            "x": "",
            "y": "",
            "hmax_pb": "",
            "hmin_pb": depth if depth is not None else "",
        })

    return rows


# --- Build combined dataframe and save ---

rows_fugro     = scan_folder(fugro_input_dir, "fugro")
rows_wiertsema = scan_folder(wiertsema_input_dir, "wiertsema")

all_rows = rows_fugro + rows_wiertsema

df_out = pd.DataFrame(all_rows)
output_csv = repo_root / "output_data" / "object_data.csv"
df_out.to_csv(output_csv, index=False)

print("Saved:", output_csv)

Saved: d:\Users\jvanruitenbeek\data_validation\output_data\object_data.csv


In [3]:
df_out

,filename,x,y,hmax_pb,hmin_pb
0,NL-2412417-HWM_B09-PB1_m_NAP_avg.csv,,,,
1,NL-2412417-HWM_B09-PB2_m_NAP_avg.csv,,,,
2,NL-2412417-HWM_B12-PB1_m_NAP_avg.csv,,,,
3,NL-2412417-HWM_B13-PB1_m_NAP_avg.csv,,,,
4,NL-2412417-HWM_B13-PB2_m_NAP_avg.csv,,,,
...,...,...,...,...,...
453,88672-1 MB650PB02 PB650-2 _F-1555.csv,,,,-16.05
454,88672-1 MB653PB01 PB653-1 _F-95.csv,,,,-1.45
455,88672-1 MB657PB01 PB657-1 _F+139.csv,,,,
456,88672-1 MB658PB01 PB658-1 _F-726.csv,,,,-7.76
